## tools


In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.document_loaders import TextLoader,PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma,FAISS
import fitz  

#utility
import numpy as np
from typing import List,Dict,Any
from sentence_transformers import SentenceTransformer

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import (RunnablePassthrough,RunnableMap)
from langchain_core.output_parsers import StrOutputParser
import os
import numpy as np
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.document_loaders import WikipediaLoader
from PIL import Image
import torch
from transformers import CLIPProcessor,CLIPModel
import base64
import io

C:\Users\kanha\AppData\Local\Temp\ipykernel_3972\1312583015.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader,PyMuPDFLoader


In [2]:
## Step 4: LLM and prompt
import os
from dotenv import load_dotenv

load_dotenv()
API=os.getenv("OPENAI_API_KEY")
llm=init_chat_model(model="groq:llama-3.3-70b-versatile",api_key=API)
response=llm.invoke("Why do parrots talk?")
response

AIMessage(content="Parrots are known for their ability to mimic human speech and other sounds, but why do they do it? There are several theories, and the answer is not a simple one. Here are some possible reasons why parrots talk:\n\n1. **Communication and Social Bonding**: In the wild, parrots use vocalizations to communicate with each other, particularly with their flock members. They may mimic sounds to establish social bonds, signal alarm, or express emotions like happiness or stress. Domesticated parrots may learn to talk as a way to bond with their human caregivers and interact with them.\n2. **Mimicry as a Survival Strategy**: In their natural habitat, parrots may mimic sounds to deceive predators, attract prey, or warn other birds of potential threats. By imitating sounds, they can create the illusion of a larger or more formidable presence, which can help protect them from harm.\n3. **Learning and Play**: Parrots are highly intelligent birds, and they have a strong instinct to

In [5]:
from langchain.tools import tool
@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It is sunny in {location}"
model_with_tool=llm.bind_tools([get_weather])

In [7]:
model_with_tool

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.0', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EA9DCD02D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EA9DE9FF90>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the wea

In [10]:
response=model_with_tool.invoke("What is the weather like in boston")
for tool_call in response.tool_calls:
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")

Tool:get_weather
Args:{'location': 'Boston'}
